In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from collections import Counter
import matplotlib.pyplot as plt
import joblib


In [2]:
#  --- BƯỚC 1: ĐỌC OUTPUT TỪ Consumer ---

df = pd.read_csv("traffic_test_output_01.csv")
df = df.sort_values("date_time").reset_index(drop=True)



## Đánh giá HMM

In [3]:
# --- ĐÁNH GIÁ GAUSSIAN ---

# bỏ các dòng warm-up chưa có state
df = df.dropna(subset=["gaussian_hmm_state"]).copy()
df["gaussian_hmm_state"] = df["gaussian_hmm_state"].astype(int)

print("=" * 70)
print("ĐÁNH GIÁ MÔ HÌNH GAUSSIAN HMM")
print("=" * 70)

# ==========================================
# 1. ĐÁNH GIÁ PHÂN BỐ TRẠNG THÁI ẨN
# ==========================================

print("\n1. ĐÁNH GIÁ PHÂN BỐ TRẠNG THÁI ẨN")
print("-" * 50)

hmm_counts = Counter(df["gaussian_hmm_state"])
total = len(df)

for state, count in sorted(hmm_counts.items()):
    pct = count / total * 100
    print(f"State {state}: {count} bản ghi ({pct:.2f}%)")

# ==========================================
# 2. ĐÁNH GIÁ ĐỘ TIN CẬY SUY DIỄN
# ==========================================

print("\n2. ĐÁNH GIÁ ĐỘ TIN CẬY SUY DIỄN")
print("-" * 50)

print(df["gaussian_hmm_prob"].describe())

mean_conf = df["gaussian_hmm_prob"].mean()
print(f"\nĐộ tin cậy trung bình: {mean_conf:.6f}")


ĐÁNH GIÁ MÔ HÌNH GAUSSIAN HMM

1. ĐÁNH GIÁ PHÂN BỐ TRẠNG THÁI ẨN
--------------------------------------------------
State 0: 686 bản ghi (13.44%)
State 1: 1773 bản ghi (34.74%)
State 2: 1641 bản ghi (32.16%)
State 3: 1003 bản ghi (19.66%)

2. ĐÁNH GIÁ ĐỘ TIN CẬY SUY DIỄN
--------------------------------------------------
count    5103.000000
mean        0.964933
std         0.095223
min         0.429607
25%         0.993054
50%         0.999496
75%         0.999929
max         1.000000
Name: gaussian_hmm_prob, dtype: float64

Độ tin cậy trung bình: 0.964933


In [4]:
# 3. ĐÁNH GIÁ TÍNH ỔN ĐỊNH THEO THỜI GIAN
# ==========================================

print("\n3. ĐÁNH GIÁ TÍNH ỔN ĐỊNH THEO THỜI GIAN")
print("-" * 50)

df["hmm_state_change"] = df["gaussian_hmm_state"].ne(
    df["gaussian_hmm_state"].shift()
).astype(int)

state_change_rate = df["hmm_state_change"].mean()

print(f"Tỷ lệ chuyển trạng thái: {state_change_rate:.4f}")

# ==========================================
# 4. ĐÁNH GIÁ ĐẶC TÍNH CHUỖI TRẠNG THÁI
# ==========================================

print("\n4. ĐÁNH GIÁ ĐẶC TÍNH CHUỖI TRẠNG THÁI")
print("-" * 50)

unique_states = df["gaussian_hmm_state"].nunique()
dominant_state = df["gaussian_hmm_state"].mode()[0]

print(f"Số trạng thái được sử dụng: {unique_states}")
print(f"Trạng thái chiếm ưu thế: State {dominant_state}")




3. ĐÁNH GIÁ TÍNH ỔN ĐỊNH THEO THỜI GIAN
--------------------------------------------------
Tỷ lệ chuyển trạng thái: 0.0941

4. ĐÁNH GIÁ ĐẶC TÍNH CHUỖI TRẠNG THÁI
--------------------------------------------------
Số trạng thái được sử dụng: 4
Trạng thái chiếm ưu thế: State 1


In [5]:
# ==========================================
# 6. LOG-LIKELIHOOD VÀ BIC (THAM KHẢO)
# ==========================================

print("\n6. ĐÁNH GIÁ LOG-LIKELIHOOD VÀ BIC (THAM KHẢO)")
print("-" * 50)

# load model + scaler
hmm_model = joblib.load("gaussian_hmm_model.pkl")
hmm_scaler = joblib.load("gaussian_hmm_scaler.pkl")

# lấy đúng feature như lúc train
feature_cols = [
    'traffic_volume', 
    'avg_24h', 
    'volume_deviation'
]

# loại NaN
df_score = df.dropna(subset=feature_cols).copy()

X_test = df_score[feature_cols].values

# scale giống training
X_test_scaled = hmm_scaler.transform(X_test)

# log-likelihood
log_likelihood = hmm_model.score(X_test_scaled)

print(f"Log-likelihood (test): {log_likelihood:.4f}")

# ==========================================
# TÍNH BIC
# ==========================================

n = len(X_test_scaled)
n_states = hmm_model.n_components
n_features = X_test_scaled.shape[1]

# số tham số gần đúng cho Gaussian HMM
k = (
    n_states * (n_states - 1) +      # transition matrix
    2 * n_states * n_features        # mean + covariance approx
)

bic = -2 * log_likelihood + k * np.log(n)

print(f"BIC (test): {bic:.4f}")


6. ĐÁNH GIÁ LOG-LIKELIHOOD VÀ BIC (THAM KHẢO)
--------------------------------------------------
Log-likelihood (test): 28928.2747
BIC (test): -57549.1964


In [6]:
# ==========================================
# 5. KẾT LUẬN TỔNG HỢP
# ==========================================

print("\n5. KẾT LUẬN TỔNG HỢP")
print("-" * 50)

# ----- đánh giá phân bố trạng thái -----
if unique_states == 1:
    state_eval = "Mô hình bị hội tụ về một trạng thái duy nhất."
elif unique_states <= 2:
    state_eval = "Mô hình sử dụng số lượng trạng thái còn hạn chế."
else:
    state_eval = f"Mô hình sử dụng đầy đủ {unique_states} trạng thái ẩn, cho thấy khả năng phân tách dữ liệu tốt."

print("• Phân bố trạng thái:", state_eval)

# ----- đánh giá confidence -----
if mean_conf < 0.6:
    conf_eval = "Độ tin cậy suy diễn thấp, mô hình chưa ổn định."
elif mean_conf < 0.8:
    conf_eval = f"Độ tin cậy suy diễn ở mức khá (mean={mean_conf:.4f})."
else:
    conf_eval = f"Độ tin cậy suy diễn cao (mean={mean_conf:.4f}), mô hình gán trạng thái ổn định."

print("• Độ tin cậy suy diễn:", conf_eval)

# ----- temporal stability -----
if state_change_rate < 0.01:
    temp_eval = "Chuỗi trạng thái quá ổn định, ít phản ánh biến động dữ liệu."
elif state_change_rate < 0.05:
    temp_eval = f"Tỷ lệ chuyển trạng thái ở mức vừa phải ({state_change_rate:.4f})."
else:
    temp_eval = f"Mô hình phản ánh được biến động theo thời gian với tỷ lệ chuyển trạng thái {state_change_rate:.4f}."

print("• Tính ổn định theo thời gian:", temp_eval)

# ----- log likelihood -----
print(
    f"• Độ phù hợp mô hình: Log-likelihood test = {log_likelihood:.4f}, "
    f"BIC = {bic:.4f}, cho thấy dữ liệu kiểm thử phù hợp tốt với mô hình đã huấn luyện."
)

print("\nKết luận chung:")
print(
    "Gaussian HMM hoạt động hiệu quả trên dữ liệu giao thông dạng chuỗi thời gian, "
    "thể hiện khả năng nhận diện trạng thái ổn định, độ tin cậy cao "
    "và mô hình hóa được động thái chuyển đổi trạng thái theo thời gian."
)

print("\n" + "=" * 70)
print("KẾT THÚC ĐÁNH GIÁ GAUSSIAN HMM")
print("=" * 70)


5. KẾT LUẬN TỔNG HỢP
--------------------------------------------------
• Phân bố trạng thái: Mô hình sử dụng đầy đủ 4 trạng thái ẩn, cho thấy khả năng phân tách dữ liệu tốt.
• Độ tin cậy suy diễn: Độ tin cậy suy diễn cao (mean=0.9649), mô hình gán trạng thái ổn định.
• Tính ổn định theo thời gian: Mô hình phản ánh được biến động theo thời gian với tỷ lệ chuyển trạng thái 0.0941.
• Độ phù hợp mô hình: Log-likelihood test = 28928.2747, BIC = -57549.1964, cho thấy dữ liệu kiểm thử phù hợp tốt với mô hình đã huấn luyện.

Kết luận chung:
Gaussian HMM hoạt động hiệu quả trên dữ liệu giao thông dạng chuỗi thời gian, thể hiện khả năng nhận diện trạng thái ổn định, độ tin cậy cao và mô hình hóa được động thái chuyển đổi trạng thái theo thời gian.

KẾT THÚC ĐÁNH GIÁ GAUSSIAN HMM


## Đánh giá K_Means

In [7]:
# ---  ĐÁNH GIÁ K- MEANS ---

df = df.dropna(subset=["kmeans_state"]).copy()
df["kmeans_state"] = df["kmeans_state"].astype(int)

print("=" * 70)
print("ĐÁNH GIÁ MÔ HÌNH KMEANS")
print("=" * 70)

# ==========================================
# 1. PHÂN BỐ CỤM
# ==========================================

print("\n1. ĐÁNH GIÁ PHÂN BỐ CỤM")
print("-" * 50)

counts = Counter(df["kmeans_state"])
total = len(df)

for state, count in sorted(counts.items()):
    pct = count / total * 100
    print(f"Cluster {state}: {count} bản ghi ({pct:.2f}%)")

# ==========================================
# 2. SILHOUETTE SCORE
# ==========================================

print("\n2. ĐÁNH GIÁ CHẤT LƯỢNG PHÂN CỤM")
print("-" * 50)

from sklearn.metrics import silhouette_score

features = df[
    ['traffic_volume', 'avg_24h', 'volume_deviation']
]

score = silhouette_score(features, df["kmeans_state"])

print(f"Silhouette Score: {score:.4f}")

# ==========================================
# 3. TEMPORAL STABILITY
# ==========================================

print("\n3. ĐÁNH GIÁ TÍNH ỔN ĐỊNH THEO THỜI GIAN")
print("-" * 50)

df["kmeans_change"] = df["kmeans_state"].ne(
    df["kmeans_state"].shift()
).astype(int)

change_rate = df["kmeans_change"].mean()

print(f"Tỷ lệ chuyển trạng thái: {change_rate:.4f}")

# ==========================================
# 4. STATE SEQUENCE INSIGHT
# ==========================================

print("\n4. ĐÁNH GIÁ ĐẶC TÍNH CHUỖI TRẠNG THÁI")
print("-" * 50)

unique_states = df["kmeans_state"].nunique()
dominant_state = df["kmeans_state"].mode()[0]

print(f"Số cụm được sử dụng: {unique_states}")
print(f"Cụm chiếm ưu thế: Cluster {dominant_state}")

# ==========================================
# 5. KẾT LUẬN
# ==========================================

print("\n5. KẾT LUẬN TỔNG HỢP")
print("-" * 50)

if unique_states == 4:
    print("• Mô hình sử dụng đầy đủ 4 cụm.")
else:
    print(f"• Mô hình sử dụng {unique_states} cụm.")

if score < 0.2:
    print("• Chất lượng phân cụm thấp.")
elif score < 0.5:
    print("• Chất lượng phân cụm ở mức trung bình.")
else:
    print("• Chất lượng phân cụm tốt.")

print(
    f"• Tỷ lệ chuyển trạng thái {change_rate:.4f}, "
    "phản ánh mức độ ổn định của chuỗi phân cụm theo thời gian."
)

print(
    "• KMeans phù hợp cho phân cụm nhanh dữ liệu giao thông, "
    "nhưng không mô hình hóa được quan hệ thời gian giữa các quan sát."
)

print("\n" + "=" * 70)
print("KẾT THÚC ĐÁNH GIÁ KMEANS")
print("=" * 70)

ĐÁNH GIÁ MÔ HÌNH KMEANS

1. ĐÁNH GIÁ PHÂN BỐ CỤM
--------------------------------------------------
Cluster 0: 2403 bản ghi (47.09%)
Cluster 1: 1007 bản ghi (19.73%)
Cluster 2: 1693 bản ghi (33.18%)

2. ĐÁNH GIÁ CHẤT LƯỢNG PHÂN CỤM
--------------------------------------------------
Silhouette Score: 0.3982

3. ĐÁNH GIÁ TÍNH ỔN ĐỊNH THEO THỜI GIAN
--------------------------------------------------
Tỷ lệ chuyển trạng thái: 0.0927

4. ĐÁNH GIÁ ĐẶC TÍNH CHUỖI TRẠNG THÁI
--------------------------------------------------
Số cụm được sử dụng: 3
Cụm chiếm ưu thế: Cluster 0

5. KẾT LUẬN TỔNG HỢP
--------------------------------------------------
• Mô hình sử dụng 3 cụm.
• Chất lượng phân cụm ở mức trung bình.
• Tỷ lệ chuyển trạng thái 0.0927, phản ánh mức độ ổn định của chuỗi phân cụm theo thời gian.
• KMeans phù hợp cho phân cụm nhanh dữ liệu giao thông, nhưng không mô hình hóa được quan hệ thời gian giữa các quan sát.

KẾT THÚC ĐÁNH GIÁ KMEANS


In [8]:
#  ÁNH GIÁ, SO SÁNH 2 MODEL --- 
# LOAD AGAIN IF NEEDED
# ==========================================

df = pd.read_csv("traffic_test_output_01.csv")

df_clean = df.dropna(subset=["gaussian_hmm_state", "kmeans_state"])

# ==========================================
# 1. STATE DISTRIBUTION COMPARISON
# ==========================================

print("\n================ STATE DISTRIBUTION COMPARISON =================")

hmm_dist = df_clean["gaussian_hmm_state"].value_counts().sort_index()
kmeans_dist = df_clean["kmeans_state"].value_counts().sort_index()

print("\nHMM:")
for k, v in hmm_dist.items():
    print(f"State {k}: {v}")

print("\nKMeans:")
for k, v in kmeans_dist.items():
    print(f"State {k}: {v}")

# ==========================================
# 2. TEMPORAL SMOOTHNESS (RẤT QUAN TRỌNG)
# ==========================================

print("\n================ TEMPORAL STABILITY =================")

hmm_change = (df_clean["gaussian_hmm_state"] != df_clean["gaussian_hmm_state"].shift()).mean()
kmeans_change = (df_clean["kmeans_state"] != df_clean["kmeans_state"].shift()).mean()

print(f"HMM state change rate: {hmm_change:.4f}")
print(f"KMeans state change rate: {kmeans_change:.4f}")

# ==========================================
# 3. UNCERTAINTY (HMM ONLY)
# ==========================================

print("\n================ CONFIDENCE =================")

print("HMM confidence:")
print(df_clean["gaussian_hmm_prob"].describe())

# ==========================================
# 4. SUMMARY COMPARISON
# ==========================================

print("\n================ FINAL COMPARISON =================")

print("""
HMM:
✔ captures temporal dependency
✔ probabilistic output
✔ smoother transitions

KMeans:
✔ simple clustering
❌ no temporal modeling
❌ no probability
""")


================ STATE DISTRIBUTION COMPARISON =================

HMM:
State 0: 686
State 1: 1773
State 2: 1641
State 3: 1003

KMeans:
State 0: 2403
State 1: 1007
State 2: 1693

================ TEMPORAL STABILITY =================
HMM state change rate: 0.0941
KMeans state change rate: 0.0927

================ CONFIDENCE =================
HMM confidence:
count    5103.000000
mean        0.964933
std         0.095223
min         0.429607
25%         0.993054
50%         0.999496
75%         0.999929
max         1.000000
Name: gaussian_hmm_prob, dtype: float64

================ FINAL COMPARISON =================

HMM:
✔ captures temporal dependency
✔ probabilistic output
✔ smoother transitions

KMeans:
✔ simple clustering
❌ no temporal modeling
❌ no probability

